In [22]:
from dotenv import load_dotenv
from pprint import pprint
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

load_dotenv()

True

In [23]:
class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [24]:
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
    })

In [25]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [26]:
response = agent.invoke(
    input={ "messages": [HumanMessage(content="My favourite colour is green")]},
    config={"configurable": {"thread_id": "1"}}
)

In [27]:
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='b718b5bd-677d-48c0-86f8-9b99b4be194e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWR3gedUhhiNzneP1IabY2PJTEn7J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da6f9-c479-7b23-b1a9-8d30be0944a2-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_wNXqXiLvZsJ7IEOmhbQDmfDN', 'type': 'tool_call'}], invalid_

In [28]:
response = agent.invoke(
    input={ 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    config={"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='b718b5bd-677d-48c0-86f8-9b99b4be194e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWR3gedUhhiNzneP1IabY2PJTEn7J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da6f9-c479-7b23-b1a9-8d30be0944a2-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_wNXqXiLvZsJ7IEOmhbQDmfDN', 'type': 'tool_call'}], invalid_

## Read state

In [17]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


In [19]:
agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [20]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='44096071-574a-47ce-9bb8-ec9fed924600'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWE4PvBYGEJZjxdwDWFsl8o78htIB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da3ff-f308-7173-af46-a5ba2b024617-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_zYg7Ja4jPZrelcsPyCGcXZ0a', 'type': 'tool_call'}], invalid_

In [21]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='44096071-574a-47ce-9bb8-ec9fed924600'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWE4PvBYGEJZjxdwDWFsl8o78htIB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da3ff-f308-7173-af46-a5ba2b024617-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_zYg7Ja4jPZrelcsPyCGcXZ0a', 'type': 'tool_call'}], invalid_

# Compare `context`, `state`, and `memory`

## Context

- context = read-only runtime data you provide for this invocation
- context is given to the agent
- Choose context when the data is already known outside the agent and should be treated as runtime dependency/input

## State

- state   = mutable agent data that can change during the run and be checkpointed
- state is owned and updated by the agent
- Choose state when the data is produced, updated, or remembered by the agent during the interaction
- The state schema defines the shape and meaning of that state (**structured conversational memory**)
- Useful for durable, structured facts:
    * Tools can read it directly.
    * You can update it intentionally.
    * You can avoid relying on the model to re-infer it.
    * You can keep it even if old messages are trimmed.
    * You can use it in deterministic program logic.

## Memory

- Combination of `checkpointer` + `config` can make an agent “remember” across calls, used to only persists the agent’s state. 
- **unstructured conversational memory**
- This is useful for conversational continuity, but it doesn't give you the structure and meaning that a state schema provides.
- Potential problems:
    * The value may be buried in a long conversation.
    * The user may contradict themselves later.
    * The model may misread the history.
    * You may trim old messages to save tokens.
    * Tools cannot reliably access the value unless they parse conversation text.